# Paper figures

Generates every `papers/seminar/fig_*.png` included by `main.tex`.

1. Run **helpers** + **data** cells.
2. Run individual figure cells, or the final **Write all** cell.

Needs: Jupyter + Pillow. Optional: local eval / attribution JSONs for live recompute.


In [ ]:
# --- Paper figure helpers (Pillow only; no matplotlib required) ---
# This cell defines paths, colors, and drawing utilities used by every figure below.
from __future__ import annotations

import json
from pathlib import Path

from IPython.display import Image, display
from PIL import Image as PILImage, ImageDraw, ImageFont

# Resolve repo root whether the notebook is opened from notebooks/ or the project root.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
elif not (ROOT / "papers" / "seminar").is_dir():
    # Fallback for kernels whose cwd is elsewhere (edit if you clone to a new path).
    ROOT = Path(r"d:\Uni\SS2026\Seminar\Project\Affordance_Benchmark")

OUT = ROOT / "papers" / "seminar"          # LaTeX includes fig_*.png from here
XAI = ROOT / "artifacts" / "report_figures" / "xai"  # qualitative heatmaps / grids
ATTR = ROOT / "artifacts" / "attribution_n100"       # vision-share + modality-gap JSON
OUT.mkdir(parents=True, exist_ok=True)

# Shared palette (RGB tuples). Keep these in sync with the "Colors / sizes" cell.
BLUE = (70, 110, 200)
CORAL = (230, 110, 90)
TEAL = (70, 160, 150)
GREY = (160, 168, 178)
INK = (40, 44, 52)
MUTED = (110, 116, 128)
GRID = (232, 234, 238)
CARD = (246, 247, 250)
WHITE = (255, 255, 255)
COL_W, COL_H = 980, 640  # default bar-chart canvas size


def show_png(path, width=720):
    """Preview a PNG in the notebook and print its path + mtime for sanity checks."""
    display(Image(filename=str(path), width=width))
    print(path, "mtime=", path.stat().st_mtime)


def _font(size, bold=False):
    """Load a TrueType font on Windows; fall back to Pillow's default bitmap font."""
    cands = []
    if bold:
        cands += [
            "C:/Windows/Fonts/segoeuib.ttf",
            "C:/Windows/Fonts/arialbd.ttf",
            "C:/Windows/Fonts/calibrib.ttf",
        ]
    cands += [
        "C:/Windows/Fonts/segoeui.ttf",
        "C:/Windows/Fonts/arial.ttf",
        "C:/Windows/Fonts/calibri.ttf",
    ]
    for name in cands:
        p = Path(name)
        if p.is_file():
            return ImageFont.truetype(str(p), size=size)
    return ImageFont.load_default()


def _tw(draw, text, font):
    """Text width in pixels (Pillow textbbox)."""
    b = draw.textbbox((0, 0), text, font=font)
    return b[2] - b[0]


def _th(draw, text, font):
    """Text height in pixels (Pillow textbbox)."""
    b = draw.textbbox((0, 0), text, font=font)
    return b[3] - b[1]


def _rounded_rect(draw, xy, radius, fill):
    """Draw a rounded rectangle; clamp radius so short sides stay valid."""
    x0, y0, x1, y1 = xy
    r = max(0, min(radius, int((x1 - x0) / 2), int((y1 - y0) / 2)))
    draw.rounded_rectangle([x0, y0, x1, y1], radius=r, fill=fill)


def _draw_ylabel(img, text, font, *, ml, mt, plot_h, color):
    """Paste a vertically rotated y-axis label into the left margin."""
    probe = ImageDraw.Draw(PILImage.new("RGB", (1, 1)))
    x0, y0, x1, y1 = probe.textbbox((0, 0), text, font=font)
    tw, th = x1 - x0, y1 - y0
    pad = 16
    layer = PILImage.new("RGBA", (tw + 2 * pad, th + 2 * pad), (0, 0, 0, 0))
    ImageDraw.Draw(layer).text((pad - x0, pad - y0), text, font=font, fill=(*color[:3], 255))
    rotated = layer.rotate(90, expand=True, resample=PILImage.Resampling.BICUBIC)
    # Center in the left margin; keep a few px clear of the canvas edge.
    x = max(6, (ml - rotated.width) // 2)
    y = mt + max(0, (plot_h - rotated.height) // 2)
    x = max(4, min(x, img.width - rotated.width - 4))
    y = max(4, min(y, img.height - rotated.height - 4))
    img.paste(rotated, (x, y), rotated)


def grouped_bars(path, labels, series, *, ylabel, ymax, width=COL_W, height=COL_H, show_values=True):
    """Grouped bar chart used for FT accuracy, fix-rate, vision share, modality gap.

    series: list of (legend_name, values_aligned_with_labels, rgb_color)
    Values are drawn as text on/above bars (zeros skipped).
    """
    left_pad = 56  # extra room so the rotated ylabel is not clipped
    img = PILImage.new("RGB", (width + left_pad, height), WHITE)
    draw = ImageDraw.Draw(img)
    tick_f = _font(26)
    label_f = _font(28, bold=True)
    y_f = _font(26, bold=True)
    leg_f = _font(20)
    val_f = _font(15, bold=True)

    n = len(labels)
    n_s = len(series)
    legend_rows = 2 if n_s >= 4 else 1  # wrap long legends (e.g. fix-rate has 4 series)
    ml, mr = left_pad + 150, 20
    mt = 26 + legend_rows * 34
    mb = 88
    plot_w = (width + left_pad) - ml - mr
    plot_h = height - mt - mb

    _rounded_rect(draw, (ml - 12, mt - 8, width + left_pad - mr + 8, mt + plot_h + 10), 12, CARD)

    # Horizontal grid + tick labels at 0/25/50/75/100% of ymax.
    for t in (0.0, 0.25, 0.5, 0.75, 1.0):
        y = mt + plot_h * (1 - t)
        val = ymax * t
        draw.line([(ml, y), (width + left_pad - mr, y)], fill=GRID, width=2)
        txt = f"{val:.2f}" if ymax <= 1.5 else f"{val:.0f}"
        draw.text((ml - 14 - _tw(draw, txt, tick_f), y - 12), txt, font=tick_f, fill=MUTED)

    draw.line([(ml, mt), (ml, mt + plot_h)], fill=INK, width=3)
    draw.line([(ml, mt + plot_h), (width + left_pad - mr, mt + plot_h)], fill=INK, width=3)
    _draw_ylabel(img, ylabel, y_f, ml=ml, mt=mt, plot_h=plot_h, color=INK)
    draw = ImageDraw.Draw(img)  # recreate after paste (draw object is stale)

    group_w = plot_w / max(n, 1)
    gap = 6 if n_s <= 4 else 4
    bar_w = max(14.0, (group_w * 0.78 - gap * (n_s - 1)) / max(n_s, 1))

    for i, lab in enumerate(labels):
        for j, (_, xs, color) in enumerate(series):
            v = float(xs[i])
            h = plot_h * (v / ymax) if ymax > 0 else 0
            x = ml + i * group_w + group_w * 0.11 + j * (bar_w + gap)
            y = mt + plot_h - h
            _rounded_rect(draw, (x, y, x + bar_w, mt + plot_h), radius=4, fill=color)
            # Flat foot so rounded tops still sit flush on the axis.
            draw.rectangle([x, mt + plot_h - 4, x + bar_w, mt + plot_h], fill=color)
            if show_values and v > 0.005:
                vt = f"{v:.2f}" if v < 2 else f"{v:.0f}"
                vtw, vth = _tw(draw, vt, val_f), _th(draw, vt, val_f)
                if h >= vth + 14:
                    # Enough room: value inside the bar (white on dark / ink on light).
                    vx = int(x + (bar_w - vtw) / 2)
                    vy = int(y + 6)
                    lum = 0.299 * color[0] + 0.587 * color[1] + 0.114 * color[2]
                    fill = WHITE if lum < 160 else INK
                    draw.text((vx, vy), vt, font=val_f, fill=fill)
                else:
                    # Short bar: value above the top edge.
                    vx = int(x + (bar_w - vtw) / 2)
                    vy = int(max(mt + 2, y - vth - 4))
                    draw.text((vx, vy), vt, font=val_f, fill=INK)
        cx = ml + (i + 0.5) * group_w
        draw.text((cx - _tw(draw, lab, label_f) / 2, height - 54), lab, font=label_f, fill=INK)

    # Legend swatches along the top.
    per_row = (n_s + legend_rows - 1) // legend_rows
    for j, (name, _, color) in enumerate(series):
        row, col = divmod(j, per_row)
        x = ml + col * (plot_w / max(per_row, 1))
        y = 10 + row * 32
        _rounded_rect(draw, (x, y + 4, x + 18, y + 22), 3, color)
        draw.text((x + 24, y), name, font=leg_f, fill=INK)

    path.parent.mkdir(parents=True, exist_ok=True)
    img.save(path, format="PNG")
    print("Wrote", path)


def stacked_panel(path, panels, *, panel_w=900, positive=None, negative=None):
    """Vertical stack of qualitative images with (a)/(b)/... badges and optional +/- captions.

    panels: list of (image_path, caption_under_panel)
    """
    def load(p):
        im = PILImage.open(p).convert("RGB")
        h = int(im.height * (panel_w / im.width))
        return im.resize((panel_w, h), PILImage.Resampling.LANCZOS)

    imgs = [load(p) for p, _ in panels]
    gap, side, top, cap_h = 18, 22, 18, 40
    caption_block_h = 78 if positive and negative else 0
    total_w = panel_w + 2 * side
    total_h = top + caption_block_h + sum(im.height + cap_h for im in imgs) + gap * (len(imgs) - 1) + 14
    canvas = PILImage.new("RGB", (total_w, total_h), WHITE)
    draw = ImageDraw.Draw(canvas)
    cap_f = _font(20, bold=True)
    badge_f = _font(22, bold=True)
    pair_f = _font(18)
    label_f = _font(18, bold=True)
    _rounded_rect(draw, (8, 8, total_w - 8, total_h - 8), 16, CARD)

    y = top
    if positive and negative:
        # Shared pos/neg caption box above the stacked panels (matches paper examples).
        box_x0, box_y0 = side, y
        box_x1, box_y1 = side + panel_w, y + caption_block_h - 10
        _rounded_rect(draw, (box_x0, box_y0, box_x1, box_y1), 12, WHITE)
        draw.rounded_rectangle([box_x0, box_y0, box_x1, box_y1], radius=12, outline=(200, 204, 210), width=2)
        draw.text((box_x0 + 14, box_y0 + 12), "+", font=label_f, fill=TEAL)
        draw.text((box_x0 + 36, box_y0 + 12), positive, font=pair_f, fill=INK)
        draw.text((box_x0 + 14, box_y0 + 40), "-", font=label_f, fill=CORAL)
        draw.text((box_x0 + 36, box_y0 + 40), negative, font=pair_f, fill=INK)
        y = box_y1 + 12

    for i, (im, (_p, cap)) in enumerate(zip(imgs, panels)):
        x = side
        canvas.paste(im, (x, y))
        draw.rectangle([x, y, x + im.width - 1, y + im.height - 1], outline=(200, 204, 210), width=2)
        _rounded_rect(draw, (x + 10, y + 10, x + 54, y + 42), 8, WHITE)
        draw.text((x + 16, y + 12), f"({chr(97 + i)})", font=badge_f, fill=INK)  # (a), (b), ...
        draw.text((x + im.width / 2 - _tw(draw, cap, cap_f) / 2, y + im.height + 10), cap, font=cap_f, fill=INK)
        y += im.height + cap_h + gap

    path.parent.mkdir(parents=True, exist_ok=True)
    canvas.save(path, format="PNG")
    print("Wrote", path, f"({total_w}x{total_h})")


def make_pipeline():
    """Write fig_pipeline.png: prefer the proposal asset, else draw a simple stage diagram."""
    asset = OUT / "assets" / "proposal_pipeline.png"
    path = OUT / "fig_pipeline.png"
    target_w = 1800  # wide enough for full \textwidth in the paper
    if asset.is_file():
        im = PILImage.open(asset).convert("RGB")
        h = int(im.height * (target_w / im.width))
        im = im.resize((target_w, h), PILImage.Resampling.LANCZOS)
        im.save(path, format="PNG")
        print("Wrote", path.name, f"({target_w}x{h})")
        return

    # Fallback diagram if the proposal asset is missing.
    stages = [
        ("PACO-LVIS", "parts + images", (45, 70, 120)),
        ("Qwen2.5-VL", "pos + hard neg", (55, 120, 75)),
        ("Human check", "pilot N=20", (130, 95, 55)),
        ("Score", "CLIP / SigLIP / Open-VLJEPA", (95, 70, 130)),
        ("Error tags", "XAI + gap", (120, 50, 60)),
    ]
    footer = (
        "Then: N=100 eval  ·  FT on ~500 and ~1000 pairs  ·  "
        "occlusion vision share  ·  embedding modality gap  ·  word heatmaps"
    )
    n = len(stages)
    cell_w, cell_h, gap = 300, 88, 28
    pad_x, pad_top, footer_h = 24, 48, 36
    total_w = pad_x * 2 + n * cell_w + (n - 1) * gap
    total_h = pad_top + cell_h + footer_h + 16
    canvas = PILImage.new("RGB", (total_w, total_h), (245, 246, 248))
    draw = ImageDraw.Draw(canvas)
    head_f, title_f, sub_f, foot_f = _font(22, True), _font(20, True), _font(15), _font(14)
    draw.text((pad_x, 12), "Project pipeline", font=head_f, fill=INK)
    for i, (title, sub, color) in enumerate(stages):
        x = pad_x + i * (cell_w + gap)
        y = pad_top
        _rounded_rect(draw, (x, y, x + cell_w, y + cell_h), 14, color)
        tw, sw = _tw(draw, title, title_f), _tw(draw, sub, sub_f)
        draw.text((x + (cell_w - tw) / 2, y + 18), title, font=title_f, fill=WHITE)
        draw.text((x + (cell_w - sw) / 2, y + 50), sub, font=sub_f, fill=(230, 232, 236))
        if i < n - 1:
            x0, x1 = x + cell_w + 3, x + cell_w + gap - 3
            mid = y + cell_h // 2
            draw.line([(x0, mid), (x1 - 8, mid)], fill=INK, width=3)
            draw.polygon([(x1, mid), (x1 - 10, mid - 6), (x1 - 10, mid + 6)], fill=INK)
    draw.text((pad_x, pad_top + cell_h + 12), footer, font=foot_f, fill=MUTED)
    canvas.save(path, format="PNG")
    print("Wrote", path.name, f"({total_w}x{total_h})")


print("ROOT:", ROOT)
print("OUT :", OUT)


## Colors / sizes
Change if you want a different look, then re-run figure cells.

In [ ]:
# Override palette here if desired, then re-run figure cells.
BLUE = (70, 110, 200)
CORAL = (230, 110, 90)
TEAL = (70, 160, 150)
GREY = (160, 168, 178)


## Data (val N=100)
Numbers match the paper tables.

In [ ]:
# Hard-coded numbers from the paper (val N=100).
# Edit here if you recompute tables, then re-run the figure cells.

FT_SCALE = {
    "labels": ["CLIP", "Open-VLJEPA"],
    "frozen / ZS": [0.63, 0.54],  # zero-shot / frozen encoders
    "FT 500": [0.87, 0.64],       # ranking fine-tune on ~500 train pairs
    "FT 1000": [0.91, 0.64],      # ranking fine-tune on ~1000 train pairs
}

# Share of ZS failures that become correct after FT, split by failure tag.
# Fractions are fixed / total_wrong_in_tag (e.g. CLIP attribute: 21 of 24).
FT_FIX = {
    "labels": ["attribute", "spatial", "fluency"],
    "CLIP FT500": [21 / 24, 7 / 8, 4 / 5],
    "CLIP FT1000": [21 / 24, 7 / 8, 4 / 5],
    "Open-VLJEPA FT500": [8 / 38, 3 / 5, 1 / 3],
    "Open-VLJEPA FT1000": [9 / 38, 4 / 5, 1 / 3],
}
print(list(FT_SCALE), list(FT_FIX))


## 1. FT scale
`fig_ft_scale.png`

In [ ]:
# fig_ft_scale.png — frozen vs FT accuracy on val N=100.
path = OUT / "fig_ft_scale.png"
grouped_bars(
    path,
    FT_SCALE["labels"],
    [
        ("frozen / ZS", FT_SCALE["frozen / ZS"], GREY),
        ("FT 500", FT_SCALE["FT 500"], BLUE),
        ("FT 1000", FT_SCALE["FT 1000"], CORAL),
    ],
    ylabel="Accuracy",
    ymax=1.0,
)
show_png(path)


## 2. Share corrected by tag
`fig_ft_fix_rate.png`

Optional: recompute CLIP FT1000 rates from eval JSON if present.

In [ ]:
# fig_ft_fix_rate.png — share of ZS errors corrected after ranking FT, by tag.
path = OUT / "fig_ft_fix_rate.png"
series = [
    ("CLIP FT500", FT_FIX["CLIP FT500"], BLUE),
    ("CLIP FT1000", FT_FIX["CLIP FT1000"], (140, 165, 220)),
    ("Open-VLJEPA FT500", FT_FIX["Open-VLJEPA FT500"], CORAL),
    ("Open-VLJEPA FT1000", FT_FIX["Open-VLJEPA FT1000"], TEAL),
]

# Optional live recompute for CLIP FT1000 if local eval dumps are present.
clip1000 = ROOT / "artifacts" / "eval" / "val_100_after_train1000" / "clip_ft.json"
if clip1000.is_file():
    import sys
    sys.path.insert(0, str(ROOT / "scripts"))
    from plot_error_tags import TAGS, tag as tag_fn

    def fix_rates(zs_path, ft_path):
        """Per-tag fix rate = (# ZS-wrong & FT-correct) / (# ZS-wrong in tag)."""
        zs = {p["image_id"]: p for p in json.loads(Path(zs_path).read_text(encoding="utf-8"))["pairs"]}
        ft = {p["image_id"]: p for p in json.loads(Path(ft_path).read_text(encoding="utf-8"))["pairs"]}
        out = []
        for tname in TAGS:
            wrong = [i for i, p in zs.items() if (not p["correct"]) and tag_fn(p) == tname]
            fixed = sum(1 for i in wrong if ft[i]["correct"])
            out.append(fixed / len(wrong) if wrong else 0.0)
        return out

    rates = fix_rates(ROOT / "humaneval" / "30jul" / "clip.json", clip1000)
    series[1] = ("CLIP FT1000", rates, (140, 165, 220))
    print("CLIP FT1000:", [round(x, 3) for x in rates])

grouped_bars(path, FT_FIX["labels"], series, ylabel="Share corrected", ymax=1.0)
show_png(path)


## 3. Pipeline
`fig_pipeline.png` (uses `papers/seminar/assets/proposal_pipeline.png` if present)

In [ ]:
# fig_pipeline.png — upscales papers/seminar/assets/proposal_pipeline.png when present.
make_pipeline()
show_png(OUT / "fig_pipeline.png")


## 4. Qualitative panels
Sources in `artifacts/report_figures/xai/`.

In [ ]:
# Spatial hard-negative example (mirror: wall vs ceiling).
path = OUT / "fig_qual_spatial.png"
stacked_panel(
    path,
    [
        (XAI / "01_spatial_clip_wall_GOOD.png", "CLIP heatmap · wall (sensible)"),
        (XAI / "02_spatial_clip_ceiling_MISGROUND.png", "CLIP heatmap · ceiling (misgrounded)"),
    ],
    panel_w=900,
    positive="Hang the mirror on the wall.",
    negative="Hang the mirror on the ceiling.",
)
show_png(path, width=520)


In [ ]:
# Attribute / action hard-negative example (microwave: open vs close).
path = OUT / "fig_qual_attribute.png"
stacked_panel(
    path,
    [
        (XAI / "06_object_clip_microwave_MISGROUND.png", "CLIP word heatmap · microwave"),
        (XAI / "09_attr_clip_microwave_grid.png", "CLIP · 3x3 occlusion grid"),
        (XAI / "10_attr_vljepa_microwave_grid.png", "Open-VLJEPA · 3x3 occlusion grid"),
    ],
    panel_w=900,
    positive="Open the microwave by pulling the door handle.",
    negative="Close the microwave by pulling the door handle.",
)
show_png(path, width=520)


## 5. XAI metrics
`fig_vision_share.png`, `fig_modality_gap.png`

Needs `artifacts/attribution_n100/modality_sensitivity.json` and `embedding_modality_gap.json`.

In [ ]:
# fig_vision_share.png — mean occlusion vision share per backend (all / correct / wrong).
sens_path = ATTR / "modality_sensitivity.json"
if sens_path.is_file():
    backends = json.loads(sens_path.read_text(encoding="utf-8")).get("backends", [])
    labels = [b["label"] for b in backends]
    path = OUT / "fig_vision_share.png"
    grouped_bars(
        path,
        labels,
        [
            ("all", [b["mean_vision_share"] for b in backends], BLUE),
            ("correct", [b["mean_vision_share_correct"] for b in backends], TEAL),
            ("wrong", [b["mean_vision_share_wrong"] for b in backends], CORAL),
        ],
        ylabel="Vision share",
        ymax=0.40,
    )
    show_png(path)
else:
    print("missing", sens_path)


In [ ]:
# fig_modality_gap.png — embedding gap + matched image–text cosines.
gap_path = ATTR / "embedding_modality_gap.json"
if gap_path.is_file():
    data = json.loads(gap_path.read_text(encoding="utf-8"))
    order = ["clip", "siglip", "open_vljepa"]
    label_map = {"clip": "CLIP", "siglip": "SigLIP", "open_vljepa": "Open-VLJEPA"}
    rows = [(label_map[k], data["backends"][k]) for k in order if k in data.get("backends", {})]
    labels = [r[0] for r in rows]
    path = OUT / "fig_modality_gap.png"
    grouped_bars(
        path,
        labels,
        [
            ("gap (pos)", [r[1]["modality_gap"] for r in rows], BLUE),
            ("gap (all)", [r[1]["modality_gap_all"] for r in rows], (140, 165, 220)),
            ("cos (pos)", [r[1]["mean_matched_cos"] for r in rows], TEAL),
            ("cos (neg)", [r[1]["mean_matched_cos_neg"] for r in rows], CORAL),
        ],
        ylabel="Gap / cosine",
        ymax=1.20,
    )
    show_png(path)
else:
    print("missing", gap_path)


## 6. Write all paper PNGs
Run after the cells above, or alone after the setup + data cells.

In [ ]:
# Regenerate every PNG the LaTeX paper includes, then delete stray paper PNGs.

# 1) Accuracy vs FT data scale
grouped_bars(
    OUT / "fig_ft_scale.png",
    FT_SCALE["labels"],
    [
        ("frozen / ZS", FT_SCALE["frozen / ZS"], GREY),
        ("FT 500", FT_SCALE["FT 500"], BLUE),
        ("FT 1000", FT_SCALE["FT 1000"], CORAL),
    ],
    ylabel="Accuracy",
    ymax=1.0,
)

# 2) Share of ZS failures corrected after FT (by error tag)
grouped_bars(
    OUT / "fig_ft_fix_rate.png",
    FT_FIX["labels"],
    [
        ("CLIP FT500", FT_FIX["CLIP FT500"], BLUE),
        ("CLIP FT1000", FT_FIX["CLIP FT1000"], (140, 165, 220)),
        ("Open-VLJEPA FT500", FT_FIX["Open-VLJEPA FT500"], CORAL),
        ("Open-VLJEPA FT1000", FT_FIX["Open-VLJEPA FT1000"], TEAL),
    ],
    ylabel="Share corrected",
    ymax=1.0,
)

# 3) Pipeline overview
make_pipeline()

# 4) Qualitative: spatial hard negative (mirror wall vs ceiling)
stacked_panel(
    OUT / "fig_qual_spatial.png",
    [
        (XAI / "01_spatial_clip_wall_GOOD.png", "CLIP heatmap · wall (sensible)"),
        (XAI / "02_spatial_clip_ceiling_MISGROUND.png", "CLIP heatmap · ceiling (misgrounded)"),
    ],
    panel_w=900,
    positive="Hang the mirror on the wall.",
    negative="Hang the mirror on the ceiling.",
)

# 5) Qualitative: attribute / action hard negative (microwave open vs close)
stacked_panel(
    OUT / "fig_qual_attribute.png",
    [
        (XAI / "06_object_clip_microwave_MISGROUND.png", "CLIP word heatmap · microwave"),
        (XAI / "09_attr_clip_microwave_grid.png", "CLIP · 3x3 occlusion grid"),
        (XAI / "10_attr_vljepa_microwave_grid.png", "Open-VLJEPA · 3x3 occlusion grid"),
    ],
    panel_w=900,
    positive="Open the microwave by pulling the door handle.",
    negative="Close the microwave by pulling the door handle.",
)

# 6) Occlusion vision share (needs artifacts/attribution_n100/modality_sensitivity.json)
sens_path = ATTR / "modality_sensitivity.json"
if sens_path.is_file():
    backends = json.loads(sens_path.read_text(encoding="utf-8")).get("backends", [])
    grouped_bars(
        OUT / "fig_vision_share.png",
        [b["label"] for b in backends],
        [
            ("all", [b["mean_vision_share"] for b in backends], BLUE),
            ("correct", [b["mean_vision_share_correct"] for b in backends], TEAL),
            ("wrong", [b["mean_vision_share_wrong"] for b in backends], CORAL),
        ],
        ylabel="Vision share",
        ymax=0.40,
    )

# 7) Embedding modality gap (needs embedding_modality_gap.json)
gap_path = ATTR / "embedding_modality_gap.json"
if gap_path.is_file():
    data = json.loads(gap_path.read_text(encoding="utf-8"))
    order = ["clip", "siglip", "open_vljepa"]
    label_map = {"clip": "CLIP", "siglip": "SigLIP", "open_vljepa": "Open-VLJEPA"}
    rows = [(label_map[k], data["backends"][k]) for k in order if k in data.get("backends", {})]
    grouped_bars(
        OUT / "fig_modality_gap.png",
        [r[0] for r in rows],
        [
            ("gap (pos)", [r[1]["modality_gap"] for r in rows], BLUE),
            ("gap (all)", [r[1]["modality_gap_all"] for r in rows], (140, 165, 220)),
            ("cos (pos)", [r[1]["mean_matched_cos"] for r in rows], TEAL),
            ("cos (neg)", [r[1]["mean_matched_cos_neg"] for r in rows], CORAL),
        ],
        ylabel="Gap / cosine",
        ymax=1.20,
    )

# Keep only PNGs that main.tex includes; remove accidental local exports.
keep = {
    "fig_ft_scale.png",
    "fig_ft_fix_rate.png",
    "fig_qual_spatial.png",
    "fig_qual_attribute.png",
    "fig_vision_share.png",
    "fig_modality_gap.png",
    "fig_pipeline.png",
}
for p in OUT.glob("*.png"):
    if p.name not in keep:
        p.unlink()
        print("Deleted", p.name)
print("Kept:", sorted(keep))


## Tips
- Open from the repo root or `notebooks/` so `ROOT` resolves.
- LaTeX reads `papers/seminar/fig_*.png`.
- Tag helpers: `scripts/plot_error_tags.py`.